In [ ]:
!pip install -q sentence-transformers faiss-cpu pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 111.8 MB/s eta 0:00:00


In [ ]:
import os
import re
import ast
import gc
import json
import math
import time
import warnings
import unicodedata
from pathlib import Path
from collections import Counter
from datetime import datetime

import numpy as np
import pandas as pd

import faiss
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 200)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
LOCAL_FAISS_DIR = Path("artifacts/faiss")
DRIVE_FAISS_DIR = Path("/content/drive/MyDrive/tablewise/artifacts_new/faiss")

if DRIVE_FAISS_DIR.exists():
    FAISS_DIR = DRIVE_FAISS_DIR
elif LOCAL_FAISS_DIR.exists():
    FAISS_DIR = LOCAL_FAISS_DIR
else:
    raise FileNotFoundError(
        "Nu am găsit artifacts/faiss."
    )

MAPPING_PATH = FAISS_DIR / "restaurant_index_mapping.parquet"
EMBEDDINGS_PATH = FAISS_DIR / "restaurant_embeddings.npy"
FAISS_INDEX_PATH = FAISS_DIR / "restaurant_faiss.index"
METADATA_PATH = FAISS_DIR / "metadata.json"

OUTPUT_DIR = FAISS_DIR.parent / "retrieval_pipeline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("FAISS_DIR:", FAISS_DIR)
print("MAPPING_PATH:", MAPPING_PATH)
print("EMBEDDINGS_PATH:", EMBEDDINGS_PATH)
print("FAISS_INDEX_PATH:", FAISS_INDEX_PATH)
print("METADATA_PATH:", METADATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

FAISS_DIR: /content/drive/MyDrive/tablewise/artifacts_new/faiss
MAPPING_PATH: /content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_index_mapping.parquet
EMBEDDINGS_PATH: /content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_embeddings.npy
FAISS_INDEX_PATH: /content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_faiss.index
METADATA_PATH: /content/drive/MyDrive/tablewise/artifacts_new/faiss/metadata.json
OUTPUT_DIR: /content/drive/MyDrive/tablewise/artifacts_new/retrieval_pipeline


In [ ]:
assert MAPPING_PATH.exists(), f"Lipsește mapping-ul: {MAPPING_PATH}"
assert EMBEDDINGS_PATH.exists(), f"Lipsesc embeddings: {EMBEDDINGS_PATH}"
assert FAISS_INDEX_PATH.exists(), f"Lipsește FAISS index: {FAISS_INDEX_PATH}"

mapping_df = pd.read_parquet(MAPPING_PATH)

embeddings = np.load(EMBEDDINGS_PATH, mmap_mode="r")

faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))

if METADATA_PATH.exists():
    with open(METADATA_PATH, "r", encoding="utf-8") as f:
        faiss_metadata = json.load(f)
else:
    faiss_metadata = {}

print("mapping_df shape:", mapping_df.shape)
print("embeddings shape:", embeddings.shape)
print("FAISS ntotal:", faiss_index.ntotal)

display(mapping_df.head(3))
faiss_metadata

mapping_df shape: (1033798, 31)
embeddings shape: (1033798, 384)
FAISS ntotal: 1033798


,faiss_idx,restaurant_id,name,country,region,province,city,city_filled,city_filled_norm,city_source,address,latitude,longitude,rating,price_level,price_bucket,top_tags_text,top_tags_norm_list,meals_text,meals_norm_list,features_text,features_norm_list,special_diets_text,special_diets_norm_list,popularity_detailed,popularity_rank_num,popularity_total_num,popularity_score,profile_quality_score,short_profile,profile_text
0,0,g10001637-d10002227,Le 147,France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,Saint-Jouvent,saint-jouvent,original_city,"10 Maison Neuve, 87510 Saint-Jouvent France",45.961674,1.169131,4.0,€,cheap,"Cheap Eats, French","[cheap eats, french]","Lunch, Dinner","[lunch, dinner]","Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Accepts Credit Cards, Table Service","[reservations, seating, wheelchair accessible, serves alcohol, accepts credit cards, table service]",Unknown,[],#1 of 2 Restaurants in Saint-Jouvent,1.0,2.0,0.36907,11,"Le 147 | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats, French","Le 147 is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 10 Maison Neuve, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French. Meals served: Lunch, Dinner. Featu..."
1,1,g10001637-d14975787,Le Saint Jouvent,France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,Saint-Jouvent,saint-jouvent,original_city,"16 Place de l Eglise, 87510 Saint-Jouvent France",45.957040,1.205480,4.0,€,cheap,Cheap Eats,[cheap eats],Unknown,[],Unknown,[],Unknown,[],#2 of 2 Restaurants in Saint-Jouvent,2.0,2.0,0.00000,9,"Le Saint Jouvent | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats","Le Saint Jouvent is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 16 Place de l Eglise, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats. Popularity: #2 of 2 Restaur..."
2,2,g10002858-d4586832,Au Bout du Pont,France,Centre-Val de Loire,Berry,Rivarennes,Rivarennes,rivarennes,original_city,"2 rue des Dames, 36800 Rivarennes France",46.635895,1.386133,5.0,€,cheap,"Cheap Eats, French, European","[cheap eats, french, european]","Dinner, Lunch, Drinks","[dinner, lunch, drinks]","Reservations, Seating, Table Service, Wheelchair Accessible","[reservations, seating, table service, wheelchair accessible]",Unknown,[],#1 of 1 Restaurant in Rivarennes,1.0,1.0,NaN,11,"Au Bout du Pont | Rivarennes, France | rating=5.0 | price=cheap | tags=Cheap Eats, French, European","Au Bout du Pont is a restaurant located in Rivarennes, Berry, Centre-Val de Loire, France. Address: 2 rue des Dames, 36800 Rivarennes France. Average rating: 5.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French, European. Meals served: Dinner, Lunch..."


{'created_at': '2026-05-04T16:00:35.667690Z',
 'input_path': '/content/drive/MyDrive/tablewise/data_new/processed/restaurants_indexable.parquet',
 'embedding_data_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurants_for_embeddings.parquet',
 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'num_restaurants': 1033798,
 'embedding_dim': 384,
 'normalized_embeddings': True,
 'faiss_index_type': 'IndexFlatIP',
 'use_sample': False,
 'sample_size': None,
 'embeddings_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_embeddings.npy',
 'faiss_index_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_faiss.index',
 'mapping_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_index_mapping.parquet',
 'python_version': '3.12.13',
 'numpy_version': '2.0.2',
 'pandas_version': '2.2.2'}

In [ ]:
required_cols = [
    "faiss_idx",
    "restaurant_id",
    "name",
    "country",
    "city_filled",
    "city_filled_norm",
    "price_bucket",
    "profile_text",
    "short_profile",
]

missing_cols = [c for c in required_cols if c not in mapping_df.columns]
assert not missing_cols, f"Lipsesc coloane obligatorii din mapping: {missing_cols}"

assert len(mapping_df) == embeddings.shape[0], "mapping_df și embeddings au dimensiuni diferite."
assert len(mapping_df) == faiss_index.ntotal, "mapping_df și FAISS index au dimensiuni diferite."
assert mapping_df["faiss_idx"].is_unique, "faiss_idx nu este unic."
assert mapping_df["faiss_idx"].min() == 0, "faiss_idx nu începe de la 0."
assert mapping_df["faiss_idx"].max() == len(mapping_df) - 1, "faiss_idx nu este continuu."

print("Validări OK.")

Validări OK.


In [ ]:
EMBEDDING_MODEL_NAME = faiss_metadata.get(
    "embedding_model",
    "sentence-transformers/all-MiniLM-L6-v2",
)

print("Loading embedding model:", EMBEDDING_MODEL_NAME)

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Model loaded.")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded.


In [ ]:
def strip_accents(text):
    if text is None or pd.isna(text):
        return ""

    text = str(text)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))

    return text


def normalize_text(x):
    if x is None or pd.isna(x):
        return ""

    s = strip_accents(str(x)).lower().strip()
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    return s


def safe_value(x, default="Unknown"):
    if x is None or pd.isna(x):
        return default

    s = str(x).strip()

    if not s or s.lower() in {"nan", "none", "null"}:
        return default

    return s


def contains_phrase(text_norm, phrase_norm):
    if not text_norm or not phrase_norm:
        return False

    pattern = rf"(?<![a-z0-9]){re.escape(phrase_norm)}(?![a-z0-9])"
    return re.search(pattern, text_norm) is not None

In [ ]:
def parse_possible_list(x):
    if x is None:
        return []

    if isinstance(x, float) and pd.isna(x):
        return []

    if isinstance(x, np.ndarray):
        values = x.tolist()
    elif isinstance(x, (list, tuple, set)):
        values = list(x)
    else:
        s = str(x).strip()

        if not s or s.lower() in {"nan", "none", "null", "unknown"}:
            return []

        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)):
                    values = list(parsed)
                else:
                    values = [s]
            except Exception:
                values = re.split(r"[,|;/]", s)
        else:
            values = re.split(r"[,|;/]", s)

    cleaned = []
    seen = set()

    for item in values:
        item_s = safe_value(item, default="").strip()
        item_norm = normalize_text(item_s)

        if item_norm and item_norm not in seen:
            cleaned.append(item_s)
            seen.add(item_norm)

    return cleaned

In [ ]:
df = mapping_df.copy()

list_base_cols = [
    "top_tags",
    "meals",
    "features",
    "special_diets",
]

for base in list_base_cols:
    text_col = f"{base}_text"
    list_col = f"{base}_list"
    norm_list_col = f"{base}_norm_list"

    if list_col not in df.columns:
        if text_col in df.columns:
            df[list_col] = df[text_col].apply(parse_possible_list)
        else:
            df[list_col] = [[] for _ in range(len(df))]
    else:
        df[list_col] = df[list_col].apply(parse_possible_list)

    if norm_list_col not in df.columns:
        df[norm_list_col] = df[list_col].apply(
            lambda xs: [normalize_text(x) for x in xs if normalize_text(x)]
        )
    else:
        df[norm_list_col] = df[norm_list_col].apply(parse_possible_list)
        df[norm_list_col] = df[norm_list_col].apply(
            lambda xs: [normalize_text(x) for x in xs if normalize_text(x)]
        )

for col in ["country", "region", "province", "city", "city_filled"]:
    if col in df.columns:
        df[f"{col}_norm"] = df[col].apply(normalize_text)

df["city_filled_norm"] = df["city_filled"].apply(normalize_text)
df["country_norm"] = df["country"].apply(normalize_text)

print("Prepared df shape:", df.shape)
display(df[["name", "country", "city_filled", "price_bucket", "top_tags_norm_list"]].head(3))

Prepared df shape: (1033798, 39)


,name,country,city_filled,price_bucket,top_tags_norm_list
0,Le 147,France,Saint-Jouvent,cheap,"[cheap eats, french]"
1,Le Saint Jouvent,France,Saint-Jouvent,cheap,[cheap eats]
2,Au Bout du Pont,France,Rivarennes,cheap,"[cheap eats, french, european]"


In [ ]:
BAD_CITY_TERMS = {
    "",
    "nan",
    "none",
    "null",
    "unknown",
    "street",
    "st",
    "road",
    "rd",
    "avenue",
    "ave",
    "square",
    "center",
    "centre",
    "park",
    "hotel",
    "restaurant",
    "restaurants",
    "best",
    "cheap",
    "budget",
    "expensive",
    "food",
    "pub",
    "bar",
    "cafe",
    "cafes",
    "coffee",
    "pizza",
    "pasta",
    "sushi",
    "seafood",
    "breakfast",
    "brunch",
    "lunch",
    "dinner",
    "europe",
    "united kingdom uk",
    "united kingdom",
    "uk",
    "england",
    "scotland",
    "wales",
    "northern ireland",
    "france",
    "italy",
    "spain",
    "germany",
    "portugal",
    "greece",
}

country_vocab = sorted(
    {
        normalize_text(x)
        for x in df["country_norm"].dropna().tolist()
        if normalize_text(x) not in {"", "unknown", "nan", "none", "null"}
    },
    key=len,
    reverse=True,
)

country_values = set(df["country_norm"].dropna().astype(str).map(normalize_text))
region_values = set(df["region_norm"].dropna().astype(str).map(normalize_text)) if "region_norm" in df.columns else set()
province_values = set(df["province_norm"].dropna().astype(str).map(normalize_text)) if "province_norm" in df.columns else set()

blocked_geo_values = country_values | region_values | province_values | BAD_CITY_TERMS

city_counts = df["city_filled_norm"].value_counts(dropna=True).to_dict()

raw_city_values = (
    df["city_filled_norm"]
    .dropna()
    .astype(str)
    .map(normalize_text)
    .tolist()
)

city_vocab = []

for city in sorted(set(raw_city_values), key=len, reverse=True):
    if not city:
        continue

    if city in blocked_geo_values:
        continue

    if len(city) < 2:
        continue

    if city.isdigit():
        continue

    if len(city.split()) > 6:
        continue

    city_vocab.append(city)

IMPORTANT_CITY_ALIASES_RAW = {
    "rome": ["rome", "roma"],
    "paris": ["paris"],
    "madrid": ["madrid"],
    "barcelona": ["barcelona"],
    "milan": ["milan", "milano"],
    "berlin": ["berlin"],
    "lisbon": ["lisbon", "lisboa"],
    "athens": ["athens", "athina", "athena"],
    "london": ["london", "londra"],
    "manchester": ["manchester"],
    "edinburgh": ["edinburgh"],
    "dublin": ["dublin"],
    "vienna": ["vienna", "wien"],
    "prague": ["prague", "praha"],
    "amsterdam": ["amsterdam"],
    "brussels": ["brussels", "bruxelles", "brussel"],
    "budapest": ["budapest"],
    "warsaw": ["warsaw", "warszawa"],
    "krakow": ["krakow", "kraków"],
    "porto": ["porto", "oporto"],
    "florence": ["florence", "firenze"],
    "venice": ["venice", "venezia"],
    "naples": ["naples", "napoli"],
    "turin": ["turin", "torino"],
    "seville": ["seville", "sevilla"],
    "valencia": ["valencia"],
    "granada": ["granada"],
    "malaga": ["malaga", "málaga"],
    "nice": ["nice"],
    "lyon": ["lyon"],
    "marseille": ["marseille"],
    "bordeaux": ["bordeaux"],
    "hamburg": ["hamburg"],
    "munich": ["munich", "muenchen", "münchen"],
    "cologne": ["cologne", "koln", "köln"],
    "frankfurt": ["frankfurt"],
}

CITY_ALIASES = {}

for canonical, aliases in IMPORTANT_CITY_ALIASES_RAW.items():
    canonical_norm = normalize_text(canonical)
    alias_norms = [normalize_text(a) for a in aliases]

    if canonical_norm not in city_vocab:
        city_vocab.append(canonical_norm)

    for alias_norm in alias_norms:
        CITY_ALIASES[alias_norm] = canonical_norm

important_order = list(IMPORTANT_CITY_ALIASES_RAW.keys())

city_vocab = sorted(
    set(city_vocab),
    key=lambda c: (
        0 if c in important_order else 1,
        important_order.index(c) if c in important_order else 999999,
        -len(c),
    ),
)

print("Countries:", len(country_vocab))
print("Cities:", len(city_vocab))

print("\nCity checks:")
for c in ["rome", "paris", "madrid", "barcelona", "milan", "berlin", "lisbon", "athens", "london"]:
    print(c, "in vocab:", c in city_vocab, "| rows:", int((df["city_filled_norm"] == c).sum()))

Countries: 24
Cities: 62251

City checks:
rome in vocab: True | rows: 12603
paris in vocab: True | rows: 18126
madrid in vocab: True | rows: 12133
barcelona in vocab: True | rows: 10283
milan in vocab: True | rows: 8381
berlin in vocab: True | rows: 0
lisbon in vocab: True | rows: 5261
athens in vocab: True | rows: 2915
london in vocab: True | rows: 0


In [ ]:
def flatten_norm_list_column(series):
    values = Counter()

    for xs in series:
        if not isinstance(xs, list):
            continue

        for x in xs:
            x_norm = normalize_text(x)

            if x_norm:
                values[x_norm] += 1

    return values


tag_counter = flatten_norm_list_column(df["top_tags_norm_list"]) if "top_tags_norm_list" in df.columns else Counter()
meal_counter = flatten_norm_list_column(df["meals_norm_list"]) if "meals_norm_list" in df.columns else Counter()
feature_counter = flatten_norm_list_column(df["features_norm_list"]) if "features_norm_list" in df.columns else Counter()
diet_counter = flatten_norm_list_column(df["special_diets_norm_list"]) if "special_diets_norm_list" in df.columns else Counter()

tag_vocab = sorted(tag_counter.keys(), key=len, reverse=True)
meal_vocab = sorted(meal_counter.keys(), key=len, reverse=True)
feature_vocab = sorted(feature_counter.keys(), key=len, reverse=True)
diet_vocab = sorted(diet_counter.keys(), key=len, reverse=True)

print("tag_vocab:", len(tag_vocab))
print("meal_vocab:", len(meal_vocab))
print("feature_vocab:", len(feature_vocab))
print("diet_vocab:", len(diet_vocab))

print("\nTop tags:")
display(pd.Series(tag_counter).sort_values(ascending=False).head(25))

print("\nTop diets:")
display(pd.Series(diet_counter).sort_values(ascending=False).head(25))

tag_vocab: 189
meal_vocab: 6
feature_vocab: 39
diet_vocab: 5

Top tags:


,0
mid-range,515029
italian,231472
cheap eats,229404
european,170832
mediterranean,151224
vegetarian friendly,125504
pizza,110138
cafe,99934
french,97358
bar,85943



Top diets:


,0
vegetarian friendly,306424
vegan options,127448
gluten free options,116841
halal,5397
kosher,249


In [ ]:
COUNTRY_ALIASES = {
    "uk": "united kingdom",
    "u k": "united kingdom",
    "great britain": "united kingdom",
    "britain": "united kingdom",
    "england": "united kingdom",
    "scotland": "united kingdom",
    "wales": "united kingdom",
    "northern ireland": "united kingdom",
    "deutschland": "germany",
    "espana": "spain",
    "españa": "spain",
    "italia": "italy",
    "grecia": "greece",
}

PRICE_KEYWORDS = {
    "cheap": [
        "cheap",
        "budget",
        "affordable",
        "low cost",
        "low-cost",
        "inexpensive",
        "not expensive",
        "economic",
        "economical",
        "ieftin",
        "ieftina",
        "ieftine",
    ],
    "mid": [
        "mid range",
        "mid-range",
        "moderate",
        "casual",
        "normal price",
        "average price",
        "medium price",
        "mediu",
        "pret mediu",
        "preț mediu",
    ],
    "expensive": [
        "expensive",
        "fine dining",
        "luxury",
        "high end",
        "high-end",
        "premium",
        "fancy",
        "upscale",
        "scump",
        "scumpa",
        "scumpe",
        "lux",
    ],
}

TAG_SYNONYMS = {
    "italian": ["italian", "italian food", "pizza", "pasta", "trattoria", "osteria"],
    "french": ["french", "french food", "bistro", "brasserie"],
    "spanish": ["spanish", "spanish food", "tapas", "paella"],
    "greek": ["greek", "greek food", "taverna", "souvlaki", "gyros"],
    "portuguese": ["portuguese", "portuguese food"],
    "german": ["german", "german food"],
    "seafood": ["seafood", "fish", "oyster", "sushi"],
    "steakhouse": ["steakhouse", "steak", "grill"],
    "asian": ["asian", "chinese", "japanese", "thai", "vietnamese", "korean"],
    "indian": ["indian", "curry"],
    "mexican": ["mexican", "tacos", "burrito"],
    "mediterranean": ["mediterranean"],
    "fast food": ["fast food", "burger", "burgers", "kebab"],
    "coffee": ["coffee", "cafe", "cafes", "café"],
    "bar": ["bar", "pub", "drinks", "cocktails"],
}

DIETARY_SYNONYMS = {
    "vegetarian friendly": ["vegetarian", "veggie", "vegetarian friendly", "fara carne", "fără carne"],
    "vegan options": ["vegan", "vegan options", "plant based", "plant-based"],
    "gluten free options": ["gluten free", "gluten-free", "without gluten", "fara gluten", "fără gluten"],
}

MEAL_SYNONYMS = {
    "breakfast": ["breakfast", "mic dejun"],
    "brunch": ["brunch"],
    "lunch": ["lunch", "pranz", "prânz"],
    "dinner": ["dinner", "cina", "cină"],
    "drinks": ["drinks", "cocktails", "beer", "wine"],
}

FEATURE_SYNONYMS = {
    "reservations": ["reservation", "reservations", "booking", "book a table", "rezervare"],
    "outdoor seating": ["outdoor", "terrace", "terrace seating", "outdoor seating", "terasă", "terasa"],
    "delivery": ["delivery", "takeaway", "take out", "takeout"],
    "wheelchair accessible": ["wheelchair", "accessible", "wheelchair accessible"],
    "family friendly": ["family friendly", "kid friendly", "kids", "children", "familie", "copii"],
    "romantic": ["romantic", "date night", "cozy", "cosy"],
}

In [ ]:
def find_synonym_matches(query_norm, synonym_dict):
    found = []

    for canonical, synonyms in synonym_dict.items():
        for syn in synonyms:
            syn_norm = normalize_text(syn)

            if contains_phrase(query_norm, syn_norm):
                found.append(canonical)
                break

    return sorted(set(found))


def extract_price_bucket(query_norm):
    for bucket, keywords in PRICE_KEYWORDS.items():
        for kw in keywords:
            kw_norm = normalize_text(kw)

            if contains_phrase(query_norm, kw_norm):
                return bucket

    return None


def extract_vocab_matches(query_norm, vocab, max_matches=8, min_len=3):
    found = []

    for term in vocab:
        term_norm = normalize_text(term)

        if len(term_norm) < min_len:
            continue

        if contains_phrase(query_norm, term_norm):
            found.append(term_norm)

        if len(found) >= max_matches:
            break

    return sorted(set(found))


def extract_country(query_norm):
    for alias, canonical in COUNTRY_ALIASES.items():
        alias_norm = normalize_text(alias)

        if contains_phrase(query_norm, alias_norm):
            canonical_norm = normalize_text(canonical)

            if canonical_norm in country_vocab:
                return canonical_norm

    for country in country_vocab:
        if contains_phrase(query_norm, country):
            return country

    return None


def city_has_location_context(query_norm, city_norm):
    if not contains_phrase(query_norm, city_norm):
        return False

    prep_patterns = [
        rf"\bin\s+{re.escape(city_norm)}\b",
        rf"\bnear\s+{re.escape(city_norm)}\b",
        rf"\baround\s+{re.escape(city_norm)}\b",
        rf"\bat\s+{re.escape(city_norm)}\b",
        rf"\bfrom\s+{re.escape(city_norm)}\b",
        rf"\bin apropiere de\s+{re.escape(city_norm)}\b",
        rf"\blanga\s+{re.escape(city_norm)}\b",
        rf"\blângă\s+{re.escape(city_norm)}\b",
        rf"\bin\s+orasul\s+{re.escape(city_norm)}\b",
        rf"\bîn\s+orasul\s+{re.escape(city_norm)}\b",
        rf"\bîn\s+orașul\s+{re.escape(city_norm)}\b",
    ]

    for pattern in prep_patterns:
        if re.search(pattern, query_norm):
            return True

    if len(city_norm) >= 4 and city_norm not in BAD_CITY_TERMS:
        return True

    return False


def extract_city(query_norm):
    for alias, canonical in CITY_ALIASES.items():
        alias_norm = normalize_text(alias)
        canonical_norm = normalize_text(canonical)

        if contains_phrase(query_norm, alias_norm) and canonical_norm in city_vocab:
            return canonical_norm

    candidates = []

    for city in city_vocab:
        if city_has_location_context(query_norm, city):
            candidates.append(city)

    if not candidates:
        return None

    return max(candidates, key=len)


def parse_user_query(query):
    query_norm = normalize_text(query)

    city = extract_city(query_norm)
    country = extract_country(query_norm)
    price_bucket = extract_price_bucket(query_norm)

    tags_from_synonyms = find_synonym_matches(query_norm, TAG_SYNONYMS)
    dietary_from_synonyms = find_synonym_matches(query_norm, DIETARY_SYNONYMS)
    meals_from_synonyms = find_synonym_matches(query_norm, MEAL_SYNONYMS)
    features_from_synonyms = find_synonym_matches(query_norm, FEATURE_SYNONYMS)

    tags_from_vocab = extract_vocab_matches(query_norm, tag_vocab, max_matches=8)
    meals_from_vocab = extract_vocab_matches(query_norm, meal_vocab, max_matches=5)
    features_from_vocab = extract_vocab_matches(query_norm, feature_vocab, max_matches=5)
    diets_from_vocab = extract_vocab_matches(query_norm, diet_vocab, max_matches=5)

    parsed = {
        "original_query": query,
        "normalized_query": query_norm,
        "city": city,
        "country": country,
        "price_bucket": price_bucket,
        "tags": sorted(set(tags_from_synonyms + tags_from_vocab)),
        "dietary": sorted(set(dietary_from_synonyms + diets_from_vocab)),
        "matched_meals": sorted(set(meals_from_synonyms + meals_from_vocab)),
        "matched_features": sorted(set(features_from_synonyms + features_from_vocab)),
    }

    return parsed

In [ ]:
test_parser_queries = [
    "cheap italian restaurant in rome",
    "best cheap restaurant in northern ireland",
    "romantic dinner in paris",
    "coffee and breakfast in athens",
    "fine dining restaurant in milan",
]

for q in test_parser_queries:
    print("=" * 120)
    print(q)
    print(json.dumps(parse_user_query(q), indent=2, ensure_ascii=False))

cheap italian restaurant in rome
{
  "original_query": "cheap italian restaurant in rome",
  "normalized_query": "cheap italian restaurant in rome",
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}
best cheap restaurant in northern ireland
{
  "original_query": "best cheap restaurant in northern ireland",
  "normalized_query": "best cheap restaurant in northern ireland",
  "city": null,
  "country": "northern ireland",
  "price_bucket": "cheap",
  "tags": [],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}
romantic dinner in paris
{
  "original_query": "romantic dinner in paris",
  "normalized_query": "romantic dinner in paris",
  "city": "paris",
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [],
  "matched_meals": [
    "dinner"
  ],
  "matched_features": [
    "romantic"
  ]
}
coffee and breakfast in athens
{
  "original_query": "

In [ ]:
def list_contains_term(row_list, term):
    if not isinstance(row_list, list):
        return False

    term_norm = normalize_text(term)

    if not term_norm:
        return False

    for item in row_list:
        item_norm = normalize_text(item)

        if not item_norm:
            continue

        if term_norm == item_norm:
            return True

        if term_norm in item_norm:
            return True

        if item_norm in term_norm and len(item_norm) >= 4:
            return True

    return False


def list_contains_any(row_list, terms):
    return any(list_contains_term(row_list, term) for term in terms)

In [ ]:
def apply_location_filter(df_in, parsed_query, verbose=True):
    current = df_in.copy()

    city = parsed_query.get("city")
    country = parsed_query.get("country")

    if city:
        before = len(current)
        filtered = current[current["city_filled_norm"] == city].copy()

        if verbose:
            print(f"City filter city={city}: {before} -> {len(filtered)}")

        if len(filtered) == 0:
            if verbose:
                print("Oraș detectat, dar fără rezultate în dataset. Returnăm subset gol.")
            return filtered

        current = filtered

    if country:
        before = len(current)
        filtered = current[current["country_norm"] == country].copy()

        if verbose:
            print(f"Country filter country={country}: {before} -> {len(filtered)}")

        if len(filtered) == 0:
            if verbose:
                print("Țară detectată, dar fără rezultate în dataset. Returnăm subset gol.")
            return filtered

        current = filtered

    return current

In [ ]:
def compute_metadata_score_for_row(row, parsed_query):
    score = 0.0
    reasons = []

    price_bucket = parsed_query.get("price_bucket")
    tags = parsed_query.get("tags", [])
    dietary = parsed_query.get("dietary", [])
    meals = parsed_query.get("matched_meals", [])
    features = parsed_query.get("matched_features", [])

    if price_bucket and row.get("price_bucket") == price_bucket:
        score += 2.0
        reasons.append(f"price={price_bucket}")

    row_tags = row.get("top_tags_norm_list", [])
    row_diets = row.get("special_diets_norm_list", [])
    row_meals = row.get("meals_norm_list", [])
    row_features = row.get("features_norm_list", [])

    for tag in tags:
        if list_contains_term(row_tags, tag):
            score += 2.0
            reasons.append(f"tag={tag}")

    for diet in dietary:
        if (
            list_contains_term(row_diets, diet)
            or list_contains_term(row_tags, diet)
            or list_contains_term(row_features, diet)
        ):
            score += 2.0
            reasons.append(f"diet={diet}")

    for meal in meals:
        if list_contains_term(row_meals, meal) or list_contains_term(row_tags, meal):
            score += 1.0
            reasons.append(f"meal={meal}")

    for feature in features:
        if list_contains_term(row_features, feature) or list_contains_term(row_tags, feature):
            score += 1.0
            reasons.append(f"feature={feature}")

    return score, sorted(set(reasons))


def add_metadata_scores(df_in, parsed_query):
    current = df_in.copy()

    if len(current) == 0:
        current["metadata_score"] = pd.Series(dtype=float)
        current["metadata_match_reasons"] = pd.Series(dtype=object)
        return current

    scores_and_reasons = current.apply(
        lambda row: compute_metadata_score_for_row(row, parsed_query),
        axis=1,
    )

    current["metadata_score"] = scores_and_reasons.apply(lambda x: x[0])
    current["metadata_match_reasons"] = scores_and_reasons.apply(lambda x: x[1])

    return current

In [ ]:
def apply_metadata_filter(df_in, parsed_query, min_results=50, verbose=True):
    current = add_metadata_scores(df_in, parsed_query)

    if len(current) == 0:
        return current

    has_constraints = bool(
        parsed_query.get("price_bucket")
        or parsed_query.get("tags")
        or parsed_query.get("dietary")
        or parsed_query.get("matched_meals")
        or parsed_query.get("matched_features")
    )

    if not has_constraints:
        if verbose:
            print("No metadata constraints.")
        return current

    before = len(current)
    filtered = current[current["metadata_score"] > 0].copy()

    if verbose:
        print(f"Metadata score > 0 filter: {before} -> {len(filtered)}")

    if len(filtered) >= min_results:
        return filtered.sort_values("metadata_score", ascending=False).reset_index(drop=True)

    if 0 < len(filtered) < min_results and before <= min_results:
        return filtered.sort_values("metadata_score", ascending=False).reset_index(drop=True)

    if verbose:
        print("Metadata filter prea strict. Păstrăm subsetul curent, dar folosim metadata_score la ranking.")

    return current.sort_values("metadata_score", ascending=False).reset_index(drop=True)

In [ ]:
def filter_candidates(query, df_in, min_metadata_results=50, verbose=True):
    parsed = parse_user_query(query)

    if verbose:
        print("Parsed query:")
        print(json.dumps(parsed, indent=2, ensure_ascii=False))
        print("\nInitial rows:", len(df_in))

    candidates = apply_location_filter(
        df_in,
        parsed,
        verbose=verbose,
    )

    if verbose:
        print("After location filter:", len(candidates))

    if len(candidates) == 0:
        candidates = add_metadata_scores(candidates, parsed)
        return parsed, candidates.reset_index(drop=True)

    candidates = apply_metadata_filter(
        candidates,
        parsed,
        min_results=min_metadata_results,
        verbose=verbose,
    )

    if verbose:
        print("After metadata filter/scoring:", len(candidates))

    return parsed, candidates.reset_index(drop=True)

In [ ]:
def encode_query(query):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    return query_embedding[0]

In [ ]:
def semantic_search_within_candidates(
    query,
    candidates_df,
    embeddings_array,
    top_k=50,
    score_chunk_size=200_000,
    verbose=True,
):
    if len(candidates_df) == 0:
        result = candidates_df.copy()
        result["semantic_score"] = pd.Series(dtype=float)
        return result

    query_embedding = encode_query(query)

    faiss_indices = candidates_df["faiss_idx"].to_numpy(dtype=np.int64)

    assert faiss_indices.min() >= 0
    assert faiss_indices.max() < embeddings_array.shape[0]

    n = len(faiss_indices)
    scores = np.empty(n, dtype=np.float32)

    if verbose:
        print(f"Computing semantic scores for {n:,} candidates...")

    for start in range(0, n, score_chunk_size):
        end = min(start + score_chunk_size, n)

        idx_chunk = faiss_indices[start:end]
        emb_chunk = embeddings_array[idx_chunk]

        scores[start:end] = emb_chunk @ query_embedding

    k = min(top_k, n)

    if k == n:
        top_positions = np.argsort(-scores)
    else:
        top_positions_unsorted = np.argpartition(-scores, k - 1)[:k]
        top_positions = top_positions_unsorted[np.argsort(-scores[top_positions_unsorted])]

    result = candidates_df.iloc[top_positions].copy().reset_index(drop=True)
    result["semantic_score"] = scores[top_positions]

    return result

In [ ]:
def add_retrieval_scores(results_df):
    results = results_df.copy()

    if len(results) == 0:
        results["metadata_score_norm"] = pd.Series(dtype=float)
        results["retrieval_score"] = pd.Series(dtype=float)
        return results

    if "metadata_score" not in results.columns:
        results["metadata_score"] = 0.0

    max_metadata = results["metadata_score"].max()

    if pd.isna(max_metadata) or max_metadata <= 0:
        results["metadata_score_norm"] = 0.0
    else:
        results["metadata_score_norm"] = results["metadata_score"] / max_metadata

    results["retrieval_score"] = (
        0.75 * results["semantic_score"].astype(float)
        + 0.25 * results["metadata_score_norm"].astype(float)
    )

    results = results.sort_values(
        ["retrieval_score", "semantic_score", "metadata_score"],
        ascending=False,
    ).reset_index(drop=True)

    return results

In [ ]:
def retrieve_restaurants(
    query,
    df_in,
    embeddings_array,
    candidate_min_metadata_results=50,
    semantic_top_k=50,
    score_chunk_size=200_000,
    verbose=True,
):
    start_time = time.time()

    parsed, candidates = filter_candidates(
        query=query,
        df_in=df_in,
        min_metadata_results=candidate_min_metadata_results,
        verbose=verbose,
    )

    if len(candidates) == 0:
        if verbose:
            print("No candidates found after filtering.")

        empty_results = candidates.copy()
        empty_results["semantic_score"] = pd.Series(dtype=float)
        empty_results["metadata_score_norm"] = pd.Series(dtype=float)
        empty_results["retrieval_score"] = pd.Series(dtype=float)

        return {
            "query": query,
            "parsed_query": parsed,
            "num_candidates_after_filter": 0,
            "results": empty_results,
            "elapsed_seconds": time.time() - start_time,
        }

    semantic_results = semantic_search_within_candidates(
        query=query,
        candidates_df=candidates,
        embeddings_array=embeddings_array,
        top_k=semantic_top_k,
        score_chunk_size=score_chunk_size,
        verbose=verbose,
    )

    final_results = add_retrieval_scores(semantic_results)

    elapsed = time.time() - start_time

    if verbose:
        print(f"Final results: {len(final_results)}")
        print(f"Elapsed seconds: {elapsed:.2f}")

    return {
        "query": query,
        "parsed_query": parsed,
        "num_candidates_after_filter": len(candidates),
        "results": final_results,
        "elapsed_seconds": elapsed,
    }

In [ ]:
def preview_retrieval_results(results_df, n=10):
    cols = [
        "faiss_idx",
        "restaurant_id",
        "name",
        "city_filled",
        "country",
        "rating",
        "price_bucket",
        "semantic_score",
        "metadata_score",
        "metadata_score_norm",
        "retrieval_score",
        "metadata_match_reasons",
        "top_tags_text",
        "special_diets_text",
        "meals_text",
        "features_text",
        "short_profile",
    ]

    cols = [c for c in cols if c in results_df.columns]

    return results_df[cols].head(n)


def pretty_print_retrieval(query, output, n=5):
    print("=" * 120)
    print("QUERY:", query)
    print("PARSED:")
    print(json.dumps(output["parsed_query"], indent=2, ensure_ascii=False))
    print("Candidates after filter:", output["num_candidates_after_filter"])
    print("Elapsed seconds:", round(output["elapsed_seconds"], 2))
    print("-" * 120)

    results = output["results"]

    if len(results) == 0:
        print("No results.")
        return

    for i, (_, row) in enumerate(results.head(n).iterrows(), start=1):
        print(f"[{i}] {row.get('name')}")
        print(f"    Location: {row.get('city_filled')}, {row.get('country')}")
        print(f"    Rating: {row.get('rating')}")
        print(f"    Price: {row.get('price_bucket')}")
        print(f"    Semantic: {row.get('semantic_score'):.4f}")
        print(f"    Metadata: {row.get('metadata_score')}")
        print(f"    Retrieval score: {row.get('retrieval_score'):.4f}")
        print(f"    Reasons: {row.get('metadata_match_reasons')}")
        print(f"    {row.get('short_profile')}")
        print()

In [ ]:
query = "cheap italian restaurant in rome"

output = retrieve_restaurants(
    query=query,
    df_in=df,
    embeddings_array=embeddings,
    candidate_min_metadata_results=50,
    semantic_top_k=30,
    verbose=True,
)

pretty_print_retrieval(query, output, n=10)
display(preview_retrieval_results(output["results"], n=20))

Parsed query:
{
  "original_query": "cheap italian restaurant in rome",
  "normalized_query": "cheap italian restaurant in rome",
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
City filter city=rome: 1033798 -> 12603
After location filter: 12603
Metadata score > 0 filter: 12603 -> 9647
After metadata filter/scoring: 9647
Computing semantic scores for 9,647 candidates...
Final results: 30
Elapsed seconds: 14.31
QUERY: cheap italian restaurant in rome
PARSED:
{
  "original_query": "cheap italian restaurant in rome",
  "normalized_query": "cheap italian restaurant in rome",
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}
Candidates after filter: 9647
Elapsed seconds: 14.31
---------------------------------------------------------------------

,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,semantic_score,metadata_score,metadata_score_norm,retrieval_score,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,673678,g187791-d7247248,Pizzeria Romaest,Rome,Italy,3.0,cheap,0.756497,4.0,1.0,0.817373,"[price=cheap, tag=italian]","Cheap Eats, Bakeries, Italian, Pizza",Unknown,"Lunch, Dinner","Takeout, Reservations, Outdoor Seating, Buffet, Seating, Parking Available, Street Parking, Television, Wheelchair Accessible, Serves Alcohol, Digital Payments, Cash Only","Pizzeria Romaest | Rome, Italy | rating=3.0 | price=cheap | tags=Cheap Eats, Bakeries, Italian, Pizza"
1,667132,g187791-d1725968,Pasta and Social International House,Rome,Italy,4.0,cheap,0.755544,4.0,1.0,0.816658,"[price=cheap, tag=italian]","Cheap Eats, Italian, Pizza, Mediterranean","Vegetarian Friendly, Vegan Options, Gluten Free Options","Lunch, Dinner",Unknown,"Pasta and Social International House | Rome, Italy | rating=4.0 | price=cheap | tags=Cheap Eats, Italian, Pizza, Mediterranean"
2,664392,g187791-d11992847,The New,Rome,Italy,4.0,cheap,0.754694,4.0,1.0,0.816020,"[price=cheap, tag=italian]","Cheap Eats, Italian, Bar, Pizza",Vegetarian Friendly,Breakfast,Unknown,"The New | Rome, Italy | rating=4.0 | price=cheap | tags=Cheap Eats, Italian, Bar, Pizza"
3,663837,g187791-d10818976,Hosteria Cacio & Pepe,Rome,Italy,4.0,cheap,0.742029,4.0,1.0,0.806522,"[price=cheap, tag=italian]","Cheap Eats, Italian, Romana, Lazio",Unknown,"Lunch, Dinner","Reservations, Seating, Serves Alcohol, Table Service, Wheelchair Accessible","Hosteria Cacio & Pepe | Rome, Italy | rating=4.0 | price=cheap | tags=Cheap Eats, Italian, Romana, Lazio"
4,666742,g187791-d15675546,Wiki Wiki Eat,Rome,Italy,3.5,cheap,0.740454,4.0,1.0,0.805340,"[price=cheap, tag=italian]","Cheap Eats, Italian, Mediterranean, Romana",Unknown,Unknown,"Seating, Serves Alcohol, Accepts Credit Cards, Table Service","Wiki Wiki Eat | Rome, Italy | rating=3.5 | price=cheap | tags=Cheap Eats, Italian, Mediterranean, Romana"
5,669786,g187791-d2373368,Pizzeria Romana,Rome,Italy,3.5,cheap,0.737781,4.0,1.0,0.803335,"[price=cheap, tag=italian]","Cheap Eats, Italian, Pizza",Unknown,"Lunch, Dinner","Reservations, Seating, Serves Alcohol, Table Service, Takeout, Outdoor Seating, Wheelchair Accessible","Pizzeria Romana | Rome, Italy | rating=3.5 | price=cheap | tags=Cheap Eats, Italian, Pizza"
6,668040,g187791-d19425254,Style,Rome,Italy,4.0,cheap,0.733098,4.0,1.0,0.799824,"[price=cheap, tag=italian]","Cheap Eats, Italian, Cafe, International",Unknown,"Breakfast, Lunch, Dinner, Drinks","Outdoor Seating, Buffet, Seating, Television, Wheelchair Accessible, Serves Alcohol, Full Bar, Wine and Beer, Digital Payments, Free Wifi, Accepts Credit Cards, Table Service, Family style","Style | Rome, Italy | rating=4.0 | price=cheap | tags=Cheap Eats, Italian, Cafe, International"
7,668038,g187791-d19424715,A Romanaccia,Rome,Italy,4.5,cheap,0.732979,4.0,1.0,0.799734,"[price=cheap, tag=italian]","Cheap Eats, Italian, Pizza, Fast food",Unknown,Unknown,Unknown,"A Romanaccia | Rome, Italy | rating=4.5 | price=cheap | tags=Cheap Eats, Italian, Pizza, Fast food"
8,672202,g187791-d4897261,La trottola roma,Rome,Italy,3.5,cheap,0.728364,4.0,1.0,0.796273,"[price=cheap, tag=italian]","Cheap Eats, Italian, Romana, Lazio",Unknown,"Lunch, Dinner","Takeout, Table Service, Seating, Wheelchair Accessible","La trottola roma | Rome, Italy | rating=3.5 | price=cheap | tags=Cheap Eats, Italian, Romana, Lazio"
9,671356,g187791-d3977981,Pizzeria Via Amerigo Vespucci Roma,Rome,Italy,4.0,cheap,0.725824,4.0,1.0,0.794368,"[price=cheap, tag=italian]","Cheap Eats, Italian, Pizza",Unknown,Dinner,Reservations,"Pizzeria Via Amerigo Vespucci Roma | Rome, Italy | rating=4.0 | price=cheap | tags=Cheap Eats, Italian, Pizza"


In [ ]:
query = "vegetarian brunch in barcelona"

output = retrieve_restaurants(
    query=query,
    df_in=df,
    embeddings_array=embeddings,
    candidate_min_metadata_results=50,
    semantic_top_k=30,
    verbose=True,
)

pretty_print_retrieval(query, output, n=10)
display(preview_retrieval_results(output["results"], n=20))

Parsed query:
{
  "original_query": "vegetarian brunch in barcelona",
  "normalized_query": "vegetarian brunch in barcelona",
  "city": "barcelona",
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [
    "brunch"
  ],
  "matched_features": []
}

Initial rows: 1033798
City filter city=barcelona: 1033798 -> 10283
After location filter: 10283
Metadata score > 0 filter: 10283 -> 3713
After metadata filter/scoring: 3713
Computing semantic scores for 3,713 candidates...
Final results: 30
Elapsed seconds: 14.10
QUERY: vegetarian brunch in barcelona
PARSED:
{
  "original_query": "vegetarian brunch in barcelona",
  "normalized_query": "vegetarian brunch in barcelona",
  "city": "barcelona",
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [
    "brunch"
  ],
  "matched_features": []
}
Candidates after filter: 3713
Elapsed seconds: 14.1
---------------

,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,semantic_score,metadata_score,metadata_score_norm,retrieval_score,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,382975,g187497-d8531379,Ugot Bruncherie,Barcelona,Spain,4.5,expensive,0.733450,3.0,1.000000,0.800087,"[diet=vegetarian friendly, meal=brunch]","Mid-range, Dessert, Cafe, Mediterranean","Vegetarian Friendly, Vegan Options, Gluten Free Options","Lunch, Brunch, Breakfast, Drinks","Highchairs Available, Wheelchair Accessible, Serves Alcohol, Accepts Mastercard, Accepts Visa, Free Wifi, Takeout, Seating, Accepts Credit Cards, Table Service","Ugot Bruncherie | Barcelona, Spain | rating=4.5 | price=expensive | tags=Mid-range, Dessert, Cafe, Mediterranean"
1,373683,g187497-d10057065,Brunch & Cake by the Sea,Barcelona,Spain,4.0,expensive,0.722497,3.0,1.000000,0.791873,"[diet=vegetarian friendly, meal=brunch]","Mid-range, American, Cafe, European","Vegetarian Friendly, Vegan Options, Gluten Free Options","Breakfast, Lunch, Brunch","Takeout, Outdoor Seating, Seating, Wheelchair Accessible, Accepts Mastercard, Accepts Visa, Accepts Credit Cards, Table Service","Brunch & Cake by the Sea | Barcelona, Spain | rating=4.0 | price=expensive | tags=Mid-range, American, Cafe, European"
2,379481,g187497-d3163747,Brunch & Cake,Barcelona,Spain,4.0,expensive,0.717026,3.0,1.000000,0.787769,"[diet=vegetarian friendly, meal=brunch]","Mid-range, Dessert, American, Cafe","Vegetarian Friendly, Vegan Options, Gluten Free Options","Breakfast, Lunch, Brunch, After-hours","Takeout, Outdoor Seating, Seating, Wheelchair Accessible, Serves Alcohol, Wine and Beer, Accepts Mastercard, Accepts Visa, Accepts Credit Cards, Table Service","Brunch & Cake | Barcelona, Spain | rating=4.0 | price=expensive | tags=Mid-range, Dessert, American, Cafe"
3,379361,g187497-d2568639,Bruc33Tapas,Barcelona,Spain,4.0,expensive,0.706748,3.0,1.000000,0.780061,"[diet=vegetarian friendly, meal=brunch]","Mid-range, Mediterranean, European, Spanish","Vegetarian Friendly, Vegan Options, Gluten Free Options","Lunch, Dinner, Brunch, After-hours","Reservations, Outdoor Seating, Seating, Wheelchair Accessible, Serves Alcohol, Full Bar, Free Wifi, Accepts Credit Cards, Table Service, Gift Cards Available","Bruc33Tapas | Barcelona, Spain | rating=4.0 | price=expensive | tags=Mid-range, Mediterranean, European, Spanish"
4,375185,g187497-d12356083,Eixampeling Brunch Café & Bar,Barcelona,Spain,5.0,mid,0.706538,3.0,1.000000,0.779903,"[diet=vegetarian friendly, meal=brunch]","Mid-range, Cafe, Fusion, Healthy","Vegetarian Friendly, Vegan Options, Gluten Free Options","Breakfast, Lunch, Brunch, Drinks",Unknown,"Eixampeling Brunch Café & Bar | Barcelona, Spain | rating=5.0 | price=mid | tags=Mid-range, Cafe, Fusion, Healthy"
5,376592,g187497-d14921698,Bar Limon Breakfast&Brunch,Barcelona,Spain,4.0,mid,0.675155,3.0,1.000000,0.756366,"[diet=vegetarian friendly, meal=brunch]","Mid-range, American, Cafe, European","Vegetarian Friendly, Vegan Options","Breakfast, Lunch, Dinner, Brunch",Unknown,"Bar Limon Breakfast&Brunch | Barcelona, Spain | rating=4.0 | price=mid | tags=Mid-range, American, Cafe, European"
6,382403,g187497-d7814939,Hummus Barcelona Vegetarian Street Food,Barcelona,Spain,4.5,mid,0.656755,3.0,1.000000,0.742566,"[diet=vegetarian friendly, meal=brunch]","Cheap Eats, Mediterranean, Middle Eastern, Vegetarian Friendly","Vegetarian Friendly, Vegan Options, Gluten Free Options","Lunch, Dinner, Brunch, Drinks","Reservations, Takeout, Outdoor Seating, Seating, Wheelchair Accessible, Serves Alcohol, Free Wifi, Accepts Credit Cards, Table Service","Hummus Barcelona Vegetarian Street Food | Barcelona, Spain | rating=4.5 | price=mid | tags=Cheap Eats, Mediterranean, Middle Eastern, Vegetarian Friendly"
7,380405,g187497-d4605459,Bo De Gracia,Barcelona,Spain,4.5,mid,0.656498,3.0,1.000000,0.742374,"[diet=vegetarian friendly, meal=brunch]","Mid-range, Seafood, Mediterranean, Europe

In [ ]:
query = "best cheap restaurant in berlin"

output = retrieve_restaurants(
    query=query,
    df_in=df,
    embeddings_array=embeddings,
    candidate_min_metadata_results=50,
    semantic_top_k=30,
    verbose=True,
)

pretty_print_retrieval(query, output, n=10)
display(preview_retrieval_results(output["results"], n=20))

Parsed query:
{
  "original_query": "best cheap restaurant in berlin",
  "normalized_query": "best cheap restaurant in berlin",
  "city": "berlin",
  "country": null,
  "price_bucket": "cheap",
  "tags": [],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
City filter city=berlin: 1033798 -> 0
Oraș detectat, dar fără rezultate în dataset. Returnăm subset gol.
After location filter: 0
No candidates found after filtering.
QUERY: best cheap restaurant in berlin
PARSED:
{
  "original_query": "best cheap restaurant in berlin",
  "normalized_query": "best cheap restaurant in berlin",
  "city": "berlin",
  "country": null,
  "price_bucket": "cheap",
  "tags": [],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}
Candidates after filter: 0
Elapsed seconds: 1.9
------------------------------------------------------------------------------------------------------------------------
No results.


,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,semantic_score,metadata_score,metadata_score_norm,retrieval_score,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile


In [ ]:
query = "best cheap restaurant in northern ireland"

output = retrieve_restaurants(
    query=query,
    df_in=df,
    embeddings_array=embeddings,
    candidate_min_metadata_results=50,
    semantic_top_k=30,
    verbose=True,
)

pretty_print_retrieval(query, output, n=10)
display(preview_retrieval_results(output["results"], n=20))

Parsed query:
{
  "original_query": "best cheap restaurant in northern ireland",
  "normalized_query": "best cheap restaurant in northern ireland",
  "city": null,
  "country": "northern ireland",
  "price_bucket": "cheap",
  "tags": [],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
Country filter country=northern ireland: 1033798 -> 2549
After location filter: 2549
Metadata score > 0 filter: 2549 -> 517
After metadata filter/scoring: 517
Computing semantic scores for 517 candidates...
Final results: 30
Elapsed seconds: 22.27
QUERY: best cheap restaurant in northern ireland
PARSED:
{
  "original_query": "best cheap restaurant in northern ireland",
  "normalized_query": "best cheap restaurant in northern ireland",
  "city": null,
  "country": "northern ireland",
  "price_bucket": "cheap",
  "tags": [],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}
Candidates after filter: 517
Elapsed seconds: 22.27
------------------------

,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,semantic_score,metadata_score,metadata_score_norm,retrieval_score,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,520632,g186477-d13991968,C.S.I Northern Ireland,Downpatrick,Northern Ireland,5.0,cheap,0.632779,2.0,1.0,0.724584,[price=cheap],"Cheap Eats, Irish, Bar, Cafe",Unknown,"Breakfast, Lunch, Dinner, Brunch, Drinks","Takeout, Buffet, Parking Available, Street Parking, Free off-street parking, Television, Wheelchair Accessible, Digital Payments, Accepts Credit Cards","C.S.I Northern Ireland | Downpatrick, Northern Ireland | rating=5.0 | price=cheap | tags=Cheap Eats, Irish, Bar, Cafe"
1,552717,g209938-d7229908,Two Chefs,Newry,Northern Ireland,3.5,cheap,0.615696,2.0,1.0,0.711772,[price=cheap],Cheap Eats,Unknown,Unknown,Unknown,"Two Chefs | Newry, Northern Ireland | rating=3.5 | price=cheap | tags=Cheap Eats"
2,520650,g186477-d5249769,Mcdonald's Restaurants,Downpatrick,Northern Ireland,2.5,cheap,0.612157,2.0,1.0,0.709118,[price=cheap],"Cheap Eats, Quick Bites, Fast food",Unknown,Breakfast,"Takeout, Wheelchair Accessible, Seating","Mcdonald's Restaurants | Downpatrick, Northern Ireland | rating=2.5 | price=cheap | tags=Cheap Eats, Quick Bites, Fast food"
3,553316,g209949-d13974679,Steamers,Templepatrick,Northern Ireland,3.0,cheap,0.608988,2.0,1.0,0.706741,[price=cheap],"Cheap Eats, Cafe",Unknown,Unknown,Unknown,"Steamers | Templepatrick, Northern Ireland | rating=3.0 | price=cheap | tags=Cheap Eats, Cafe"
4,553546,g209954-d10844638,Neptune's Larder Cafe,Kilkeel,Northern Ireland,5.0,cheap,0.602738,2.0,1.0,0.702053,[price=cheap],Cheap Eats,Unknown,Unknown,Unknown,"Neptune's Larder Cafe | Kilkeel, Northern Ireland | rating=5.0 | price=cheap | tags=Cheap Eats"
5,552640,g209938-d12850616,The Full Irish,Newry,Northern Ireland,4.5,cheap,0.602614,2.0,1.0,0.701960,[price=cheap],"Cheap Eats, Irish",Unknown,Unknown,Unknown,"The Full Irish | Newry, Northern Ireland | rating=4.5 | price=cheap | tags=Cheap Eats, Irish"
6,520944,g186484-d6878817,McDonald's Omagh - Northern Ireland,Omagh,Northern Ireland,2.5,cheap,0.596644,2.0,1.0,0.697483,[price=cheap],"Cheap Eats, Quick Bites, Fast food",Unknown,Breakfast,"Takeout, Seating, Free Wifi, Wheelchair Accessible","McDonald's Omagh - Northern Ireland | Omagh, Northern Ireland | rating=2.5 | price=cheap | tags=Cheap Eats, Quick Bites, Fast food"
7,520846,g186482-d4861407,The Cake Shop Derry & Donegal,Derry,Northern Ireland,4.0,cheap,0.593840,2.0,1.0,0.695380,[price=cheap],"Cheap Eats, Irish, Cafe",Unknown,Unknown,Unknown,"The Cake Shop Derry & Donegal | Derry, Northern Ireland | rating=4.0 | price=cheap | tags=Cheap Eats, Irish, Cafe"
8,600036,g667786-d5539648,Silver Lounge,Larne,Northern Ireland,4.5,cheap,0.593454,2.0,1.0,0.695091,[price=cheap],"Cheap Eats, Cafe, British, Grill",Vegetarian Friendly,"Breakfast, Lunch, Dinner, Brunch",Unknown,"Silver Lounge | Larne, Northern Ireland | rating=4.5 | price=cheap | tags=Cheap Eats, Cafe, British, Grill"
9,520871,g186482-d7691747,Mason's Bar,Derry,Northern Ireland,4.5,cheap,0.590894,2.0,1.0,0.693171,[price=cheap],Cheap Eats,Unknown,Unknown,Unknown,"Mason's Bar | Derry, Northern Ireland | rating=4.5 | price=cheap | tags=Cheap Eats"


In [ ]:
test_queries = [
    "cheap italian restaurant in rome",
    "vegetarian brunch in barcelona",
    "romantic dinner in paris",
    "family friendly seafood place in lisbon",
    "coffee and breakfast in athens",
    "expensive fine dining in milan",
    "tapas place in madrid",
    "gluten free restaurant in paris",
]

batch_rows = []
batch_outputs = {}

for q in test_queries:
    print("=" * 120)
    print("Running:", q)

    out = retrieve_restaurants(
        query=q,
        df_in=df,
        embeddings_array=embeddings,
        candidate_min_metadata_results=50,
        semantic_top_k=20,
        verbose=False,
    )

    batch_outputs[q] = out

    results = out["results"]

    top_row = results.iloc[0] if len(results) > 0 else {}

    batch_rows.append({
        "query": q,
        "parsed_city": out["parsed_query"].get("city"),
        "parsed_country": out["parsed_query"].get("country"),
        "parsed_price_bucket": out["parsed_query"].get("price_bucket"),
        "parsed_tags": out["parsed_query"].get("tags"),
        "parsed_dietary": out["parsed_query"].get("dietary"),
        "parsed_meals": out["parsed_query"].get("matched_meals"),
        "parsed_features": out["parsed_query"].get("matched_features"),
        "num_candidates_after_filter": out["num_candidates_after_filter"],
        "num_results": len(results),
        "top_name": top_row.get("name", None) if len(results) > 0 else None,
        "top_city": top_row.get("city_filled", None) if len(results) > 0 else None,
        "top_country": top_row.get("country", None) if len(results) > 0 else None,
        "top_price": top_row.get("price_bucket", None) if len(results) > 0 else None,
        "top_semantic_score": top_row.get("semantic_score", None) if len(results) > 0 else None,
        "top_metadata_score": top_row.get("metadata_score", None) if len(results) > 0 else None,
        "top_retrieval_score": top_row.get("retrieval_score", None) if len(results) > 0 else None,
        "elapsed_seconds": out["elapsed_seconds"],
    })

batch_summary_df = pd.DataFrame(batch_rows)
display(batch_summary_df)

Running: cheap italian restaurant in rome
Running: vegetarian brunch in barcelona
Running: romantic dinner in paris
Running: family friendly seafood place in lisbon
Running: coffee and breakfast in athens
Running: expensive fine dining in milan
Running: tapas place in madrid
Running: gluten free restaurant in paris


,query,parsed_city,parsed_country,parsed_price_bucket,parsed_tags,parsed_dietary,parsed_meals,parsed_features,num_candidates_after_filter,num_results,top_name,top_city,top_country,top_price,top_semantic_score,top_metadata_score,top_retrieval_score,elapsed_seconds
0,cheap italian restaurant in rome,rome,None,cheap,[italian],[],[],[],9647,20,Pizzeria Romaest,Rome,Italy,cheap,0.756497,4.0,0.817373,2.292908
1,vegetarian brunch in barcelona,barcelona,None,None,[],[vegetarian friendly],[brunch],[],3713,20,Ugot Bruncherie,Barcelona,Spain,expensive,0.733450,3.0,0.800087,2.568729
2,romantic dinner in paris,paris,None,None,[],[],[dinner],[romantic],9307,20,Romantica Caffe Etoile,Paris,France,expensive,0.605484,1.0,0.704113,3.863931
3,family friendly seafood place in lisbon,lisbon,None,None,[seafood],[],[],[family friendly],286,20,Traditional Portuguese and Sea Food Restaurant,Lisbon,Portugal,expensive,0.692461,2.0,0.769346,4.607907
4,coffee and breakfast in athens,athens,None,None,[coffee],[],[breakfast],[],580,20,Coffee House Food & Drink,Athens,Greece,unknown,0.710915,1.0,0.783186,4.635519
5,expensive fine dining in milan,milan,None,expensive,[fine dining],[],[],[],2012,20,My Kitchen Club,Milan,Italy,expensive,0.673239,4.0,0.754929,2.188441
6,tapas place in madrid,madrid,None,None,[spanish],[],[],[],5305,20,Tapa Tapa,Madrid,Spain,mid,0.839951,2.0,0.879963,5.730106
7,gluten free restaurant in paris,paris,None,None,[],[gluten free options],[],[],584,20,Restaurant H,Paris,France,expensive,0.735871,2.0,0.801903,2.755863


In [ ]:
location_diagnostic_rows = []

for q, out in batch_outputs.items():
    parsed_city = out["parsed_query"].get("city")
    parsed_country = out["parsed_query"].get("country")
    results = out["results"]

    if len(results) == 0:
        city_ok_rate = None
        country_ok_rate = None
    else:
        if parsed_city:
            city_ok_rate = float((results["city_filled_norm"] == parsed_city).mean())
        else:
            city_ok_rate = None

        if parsed_country:
            country_ok_rate = float((results["country_norm"] == parsed_country).mean())
        else:
            country_ok_rate = None

    location_diagnostic_rows.append({
        "query": q,
        "parsed_city": parsed_city,
        "parsed_country": parsed_country,
        "num_results": len(results),
        "city_ok_rate": city_ok_rate,
        "country_ok_rate": country_ok_rate,
    })

location_diagnostic_df = pd.DataFrame(location_diagnostic_rows)
display(location_diagnostic_df)

,query,parsed_city,parsed_country,num_results,city_ok_rate,country_ok_rate
0,cheap italian restaurant in rome,rome,None,20,1.0,None
1,vegetarian brunch in barcelona,barcelona,None,20,1.0,None
2,romantic dinner in paris,paris,None,20,1.0,None
3,family friendly seafood place in lisbon,lisbon,None,20,1.0,None
4,coffee and breakfast in athens,athens,None,20,1.0,None
5,expensive fine dining in milan,milan,None,20,1.0,None
6,tapas place in madrid,madrid,None,20,1.0,None
7,gluten free restaurant in paris,paris,None,20,1.0,None


In [ ]:
price_diagnostic_rows = []

for q, out in batch_outputs.items():
    parsed_price = out["parsed_query"].get("price_bucket")
    results = out["results"]

    if len(results) == 0 or not parsed_price:
        price_ok_rate = None
        top10_price_ok_rate = None
    else:
        price_ok_rate = float((results["price_bucket"] == parsed_price).mean())
        top10_price_ok_rate = float((results.head(10)["price_bucket"] == parsed_price).mean())

    price_diagnostic_rows.append({
        "query": q,
        "parsed_price_bucket": parsed_price,
        "num_results": len(results),
        "price_ok_rate_all_results": price_ok_rate,
        "price_ok_rate_top10": top10_price_ok_rate,
    })

price_diagnostic_df = pd.DataFrame(price_diagnostic_rows)
display(price_diagnostic_df)

,query,parsed_price_bucket,num_results,price_ok_rate_all_results,price_ok_rate_top10
0,cheap italian restaurant in rome,cheap,20,0.6,0.9
1,vegetarian brunch in barcelona,None,20,NaN,NaN
2,romantic dinner in paris,None,20,NaN,NaN
3,family friendly seafood place in lisbon,None,20,NaN,NaN
4,coffee and breakfast in athens,None,20,NaN,NaN
5,expensive fine dining in milan,expensive,20,1.0,1.0
6,tapas place in madrid,None,20,NaN,NaN
7,gluten free restaurant in paris,None,20,NaN,NaN


In [ ]:
def full_faiss_search(query, top_k=10):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    scores, indices = faiss_index.search(query_embedding, top_k)

    result = df.iloc[indices[0]].copy().reset_index(drop=True)
    result["semantic_score"] = scores[0]

    return result


comparison_query = "vegetarian restaurant in berlin"

print("Full FAISS brute search:")
full_results = full_faiss_search(comparison_query, top_k=10)
display(preview_retrieval_results(full_results, n=10))

print("\nFiltered retrieval pipeline:")
filtered_output = retrieve_restaurants(
    query=comparison_query,
    df_in=df,
    embeddings_array=embeddings,
    candidate_min_metadata_results=50,
    semantic_top_k=10,
    verbose=False,
)
display(preview_retrieval_results(filtered_output["results"], n=10))

Full FAISS brute search:


,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,semantic_score,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,68149,g187198-d11698734,Berlin 1989,Nantes,France,4.5,mid,0.692160,"Cheap Eats, German, European, Vegetarian Friendly",Vegetarian Friendly,Unknown,Unknown,"Berlin 1989 | Nantes, France | rating=4.5 | price=mid | tags=Cheap Eats, German, European, Vegetarian Friendly"
1,987648,g274707-d21166531,Berlin Street Food,Prague,Czech Republic,5.0,cheap,0.688706,"Cheap Eats, Turkish",Unknown,Lunch,Unknown,"Berlin Street Food | Prague, Czech Republic | rating=5.0 | price=cheap | tags=Cheap Eats, Turkish"
2,214809,g187310-d12000161,Veganel,Nuremberg,Germany,4.5,cheap,0.681258,"Cheap Eats, Pizza, Fast food, Vegetarian Friendly","Vegetarian Friendly, Vegan Options","Lunch, Dinner, Drinks",Unknown,"Veganel | Nuremberg, Germany | rating=4.5 | price=cheap | tags=Cheap Eats, Pizza, Fast food, Vegetarian Friendly"
3,188183,g187371-d2390072,Well Being Vegetarian,Cologne,Germany,4.0,mid,0.680317,"Mid-range, Asian, Vietnamese, Vegetarian Friendly","Vegetarian Friendly, Gluten Free Options, Vegan Options","Lunch, Dinner",Unknown,"Well Being Vegetarian | Cologne, Germany | rating=4.0 | price=mid | tags=Mid-range, Asian, Vietnamese, Vegetarian Friendly"
4,188453,g187371-d5483057,Restaurant Zagreb,Cologne,Germany,4.5,mid,0.673349,"Mid-range, European, Croatian, Vegetarian Friendly",Vegetarian Friendly,"Lunch, Dinner",Unknown,"Restaurant Zagreb | Cologne, Germany | rating=4.5 | price=mid | tags=Mid-range, European, Croatian, Vegetarian Friendly"
5,209060,g187291-d4182478,[m]eatery bar + restaurant,Stuttgart,Germany,4.0,expensive,0.671303,"Fine Dining, Steakhouse, European, Grill","Vegetarian Friendly, Gluten Free Options",Unknown,Unknown,"[m]eatery bar + restaurant | Stuttgart, Germany | rating=4.0 | price=expensive | tags=Fine Dining, Steakhouse, European, Grill"
6,367975,g187481-d10667214,Cafeteria Berlin,Puerto de la Cruz,Spain,4.0,cheap,0.670935,"Cheap Eats, Mediterranean, Spanish, Vegetarian Friendly",Vegetarian Friendly,Unknown,"Reservations, Table Service, Seating","Cafeteria Berlin | Puerto de la Cruz, Spain | rating=4.0 | price=cheap | tags=Cheap Eats, Mediterranean, Spanish, Vegetarian Friendly"
7,214788,g187310-d11810431,Restaurante Siculum,Nuremberg,Germany,4.5,expensive,0.667995,"Fine Dining, Italian, Mediterranean, Vegetarian Friendly","Vegetarian Friendly, Vegan Options","Dinner, Lunch",Unknown,"Restaurante Siculum | Nuremberg, Germany | rating=4.5 | price=expensive | tags=Fine Dining, Italian, Mediterranean, Vegetarian Friendly"
8,227228,g187367-d17068234,Restaurant Sammy's,Aachen,Germany,5.0,expensive,0.663719,"Fine Dining, European, Central European, Vegetarian Friendly",Vegetarian Friendly,"Dinner, Lunch, Drinks",Unknown,"Restaurant Sammy's | Aachen, Germany | rating=5.0 | price=expensive | tags=Fine Dining, European, Central European, Vegetarian Friendly"
9,1024013,g1088372-d2306420,Hotel Restaurant Berlin Beach,Sveti Vlas,Bulgaria,4.5,mid,0.663613,"Mid-range, Vegetarian Friendly",Vegetarian Friendly,"Lunch, Dinner","Reservations, Seating, Table Service","Hotel Restaurant Berlin Beach | Sveti Vlas, Bulgaria | rating=4.5 | price=mid | tags=Mid-range, Vegetarian Friendly"



Filtered retrieval pipeline:


,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,semantic_score,metadata_score,metadata_score_norm,retrieval_score,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,68149,g187198-d11698734,Berlin 1989,Nantes,France,4.5,mid,0.692160,2.0,1.0,0.769120,[diet=vegetarian friendly],"Cheap Eats, German, European, Vegetarian Friendly",Vegetarian Friendly,Unknown,Unknown,"Berlin 1989 | Nantes, France | rating=4.5 | price=mid | tags=Cheap Eats, German, European, Vegetarian Friendly"
1,214809,g187310-d12000161,Veganel,Nuremberg,Germany,4.5,cheap,0.681258,2.0,1.0,0.760944,[diet=vegetarian friendly],"Cheap Eats, Pizza, Fast food, Vegetarian Friendly","Vegetarian Friendly, Vegan Options","Lunch, Dinner, Drinks",Unknown,"Veganel | Nuremberg, Germany | rating=4.5 | price=cheap | tags=Cheap Eats, Pizza, Fast food, Vegetarian Friendly"
2,188183,g187371-d2390072,Well Being Vegetarian,Cologne,Germany,4.0,mid,0.680317,2.0,1.0,0.760237,[diet=vegetarian friendly],"Mid-range, Asian, Vietnamese, Vegetarian Friendly","Vegetarian Friendly, Gluten Free Options, Vegan Options","Lunch, Dinner",Unknown,"Well Being Vegetarian | Cologne, Germany | rating=4.0 | price=mid | tags=Mid-range, Asian, Vietnamese, Vegetarian Friendly"
3,188453,g187371-d5483057,Restaurant Zagreb,Cologne,Germany,4.5,mid,0.673349,2.0,1.0,0.755012,[diet=vegetarian friendly],"Mid-range, European, Croatian, Vegetarian Friendly",Vegetarian Friendly,"Lunch, Dinner",Unknown,"Restaurant Zagreb | Cologne, Germany | rating=4.5 | price=mid | tags=Mid-range, European, Croatian, Vegetarian Friendly"
4,209060,g187291-d4182478,[m]eatery bar + restaurant,Stuttgart,Germany,4.0,expensive,0.671303,2.0,1.0,0.753477,[diet=vegetarian friendly],"Fine Dining, Steakhouse, European, Grill","Vegetarian Friendly, Gluten Free Options",Unknown,Unknown,"[m]eatery bar + restaurant | Stuttgart, Germany | rating=4.0 | price=expensive | tags=Fine Dining, Steakhouse, European, Grill"
5,367975,g187481-d10667214,Cafeteria Berlin,Puerto de la Cruz,Spain,4.0,cheap,0.670935,2.0,1.0,0.753201,[diet=vegetarian friendly],"Cheap Eats, Mediterranean, Spanish, Vegetarian Friendly",Vegetarian Friendly,Unknown,"Reservations, Table Service, Seating","Cafeteria Berlin | Puerto de la Cruz, Spain | rating=4.0 | price=cheap | tags=Cheap Eats, Mediterranean, Spanish, Vegetarian Friendly"
6,214788,g187310-d11810431,Restaurante Siculum,Nuremberg,Germany,4.5,expensive,0.667995,2.0,1.0,0.750997,[diet=vegetarian friendly],"Fine Dining, Italian, Mediterranean, Vegetarian Friendly","Vegetarian Friendly, Vegan Options","Dinner, Lunch",Unknown,"Restaurante Siculum | Nuremberg, Germany | rating=4.5 | price=expensive | tags=Fine Dining, Italian, Mediterranean, Vegetarian Friendly"
7,227228,g187367-d17068234,Restaurant Sammy's,Aachen,Germany,5.0,expensive,0.663719,2.0,1.0,0.747789,[diet=vegetarian friendly],"Fine Dining, European, Central European, Vegetarian Friendly",Vegetarian Friendly,"Dinner, Lunch, Drinks",Unknown,"Restaurant Sammy's | Aachen, Germany | rating=5.0 | price=expensive | tags=Fine Dining, European, Central European, Vegetarian Friendly"
8,1024013,g1088372-d2306420,Hotel Restaurant Berlin Beach,Sveti Vlas,Bulgaria,4.5,mid,0.663613,2.0,1.0,0.747710,[diet=vegetarian friendly],"Mid-range, Vegetarian Friendly",Vegetarian Friendly,"Lunch, Dinner","Reservations, Seating, Table Service","Hotel Restaurant Berlin Beach | Sveti Vlas, Bulgaria | rating=4.5 | price=mid | tags=Mid-range, Vegetarian Friendly"
9,238435,g187399-d8095523,[m]eatery Bar + Restaurant,Dresden,Germany,4.0,expensive,0.662704,2.0,1.0,0.747028,[diet=vegetarian friendly],"Fine Dining, Steakhouse, Barbecue, European",Vegetarian Friendly,Unknown,Unknown,"[m]eatery Bar + Restaurant | Dresden, Germany | rating=4.0 | price=expensive | tags=Fine Dining, Steakhouse, Barbecue, European"


In [ ]:
BATCH_SUMMARY_PATH = OUTPUT_DIR / "retrieval_batch_summary.csv"
batch_summary_df.to_csv(BATCH_SUMMARY_PATH, index=False)

LOCATION_DIAGNOSTIC_PATH = OUTPUT_DIR / "retrieval_location_diagnostics.csv"
location_diagnostic_df.to_csv(LOCATION_DIAGNOSTIC_PATH, index=False)

PRICE_DIAGNOSTIC_PATH = OUTPUT_DIR / "retrieval_price_diagnostics.csv"
price_diagnostic_df.to_csv(PRICE_DIAGNIC_PATH if False else PRICE_DIAGNOSTIC_PATH, index=False)

print("Saved:", BATCH_SUMMARY_PATH)
print("Saved:", LOCATION_DIAGNOSTIC_PATH)
print("Saved:", PRICE_DIAGNOSTIC_PATH)

Saved: /content/drive/MyDrive/tablewise/artifacts_new/retrieval_pipeline/retrieval_batch_summary.csv
Saved: /content/drive/MyDrive/tablewise/artifacts_new/retrieval_pipeline/retrieval_location_diagnostics.csv
Saved: /content/drive/MyDrive/tablewise/artifacts_new/retrieval_pipeline/retrieval_price_diagnostics.csv


In [ ]:
example_results_dir = OUTPUT_DIR / "example_results"
example_results_dir.mkdir(parents=True, exist_ok=True)

for q, out in batch_outputs.items():
    safe_name = normalize_text(q).replace(" ", "_")[:80]
    path = example_results_dir / f"{safe_name}.csv"

    results_to_save = out["results"].copy()
    results_to_save["query"] = q
    results_to_save["parsed_query_json"] = json.dumps(out["parsed_query"], ensure_ascii=False)

    results_to_save.to_csv(path, index=False)

print("Saved example result CSVs in:", example_results_dir)

Saved example result CSVs in: /content/drive/MyDrive/tablewise/artifacts_new/retrieval_pipeline/example_results


In [ ]:
retrieval_config = {
    "created_at": datetime.utcnow().isoformat() + "Z",
    "faiss_dir": str(FAISS_DIR),
    "mapping_path": str(MAPPING_PATH),
    "embeddings_path": str(EMBEDDINGS_PATH),
    "faiss_index_path": str(FAISS_INDEX_PATH),
    "embedding_model": EMBEDDING_MODEL_NAME,
    "num_restaurants": int(len(df)),
    "embedding_dim": int(embeddings.shape[1]),
    "city_vocab_size": int(len(city_vocab)),
    "country_vocab_size": int(len(country_vocab)),
    "tag_vocab_size": int(len(tag_vocab)),
    "meal_vocab_size": int(len(meal_vocab)),
    "feature_vocab_size": int(len(feature_vocab)),
    "diet_vocab_size": int(len(diet_vocab)),
    "retrieval_score_formula": "0.75 * semantic_score + 0.25 * metadata_score_norm",
    "location_filter_policy": "city and country are hard filters; no fallback to all Europe when explicit city has zero results",
}

RETRIEVAL_CONFIG_PATH = OUTPUT_DIR / "retrieval_config.json"

with open(RETRIEVAL_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(retrieval_config, f, indent=2, ensure_ascii=False)

print("Saved retrieval config:", RETRIEVAL_CONFIG_PATH)
retrieval_config

Saved retrieval config: /content/drive/MyDrive/tablewise/artifacts_new/retrieval_pipeline/retrieval_config.json


{'created_at': '2026-05-04T16:55:43.156300Z',
 'faiss_dir': '/content/drive/MyDrive/tablewise/artifacts_new/faiss',
 'mapping_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_index_mapping.parquet',
 'embeddings_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_embeddings.npy',
 'faiss_index_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_faiss.index',
 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'num_restaurants': 1033798,
 'embedding_dim': 384,
 'city_vocab_size': 62246,
 'country_vocab_size': 24,
 'tag_vocab_size': 189,
 'meal_vocab_size': 6,
 'feature_vocab_size': 39,
 'diet_vocab_size': 5,
 'retrieval_score_formula': '0.75 * semantic_score + 0.25 * metadata_score_norm',
 'location_filter_policy': 'city and country are hard filters; no fallback to all Europe when explicit city has zero results'}

In [ ]:
def tablewise_retrieve_candidates(
    query,
    top_k=50,
    verbose=False,
):
    return retrieve_restaurants(
        query=query,
        df_in=df,
        embeddings_array=embeddings,
        candidate_min_metadata_results=50,
        semantic_top_k=top_k,
        score_chunk_size=200_000,
        verbose=verbose,
    )


def tablewise_preview_candidates(query, top_k=20):
    out = tablewise_retrieve_candidates(
        query=query,
        top_k=top_k,
        verbose=True,
    )

    display(preview_retrieval_results(out["results"], n=top_k))

    return out